- Optuna:
    - It is a hyperparameter tuning framework.
    - It can be used for hyperparemeter tuning of large datasets.

- Why Optuna :
    - we have students data :

        |    Cgpa   |   iq   |   placed   |
        |-----------|--------|------------|
        |     8     |   78   |     1      |
        |     5     |  110   |     0      |
        |     7     |  101   |     1      |
        |     9     |   89   |     1      |
        |     6     |   86   |     0      |
        
    - This clssification problem to solve this we will use reandom forest
    - We have to train our model in way that accuracy will be maximum
    - In random forest there are multiple hyperparameters but we are focusing on max_depth and n_estimators
    - so now our problem statement becomes to find out the values of max_depth and n_estimators so we can get maximum accuracy.
    - based on our intution we have to define search space on that all values will be tested, basically for n_estimator 1 to 5 (it will search for all values from 1 to 5), similarly n_estimator(50,250,step sep size will be 10)
    - search space :
                  
        | max_depth \ n_estimators | 50 | 100 | 150 | 200 | 250 |
        |--------------------------|----|-----|-----|-----|-----|
        | 1                        |    |     |     |     |     |
        | 2                        |    |     |     |     |     |
        | 3                        |    |     |     |     |     |
        | 4                        |    |     |     |     |     |
        | 5                        |    |     |     |     |     |
    -  We have to try 5 values of n_estimator and 5 values of max_depth and all combinations of them, in a way we have grid.
    - for this firstly gridsearch come in picture, it will try all combination, and this is very costly
    - to solve this random search come in picture it try random cobinations, and sometime this will not give best parameter values
    - In random search and grid search we have problems, so optuna now came in picture
    - Optuna use different type of search, this is 'Bayesian', it is mathetical way of searching inspired by probability
    - Bayesian search is worked on a principle, suppose for this problem it max_depth,n_estimator change lead to change of accuracy so there is mathematical realtion between max_depth, n_estimator and accuracy. Bayesian search esablish a fact that $accuracy = f(max_depth,n_estimator)$
    - Bayesian search try to find out type of relationship. And based on this try to select values of hyperparameter and try to select search value.

- Key Terms:
    1) Study:
        - A Study is the entire hyperparameter optimization process in Optuna.
        - Its goal is to find the best set of hyperparameters for a model.
        - A Study consists of multiple Trials.
        - It keeps track of all trials, their hyperparameters, and their scores.
    2) Trail:
        - It is single itration of optimaztion process or study
        - In each Trial, Optuna selects a set of hyperparameters,trains the model, evaluates it, and records the result.
    3) Trail Parametres:
        - These are specific hyperparameters values chosed on each trail(itration)
    4) Objective Function:
        - Objective function is a relation that we are trying to find out
    5) Sampler:
        - It is main component that is useful for selcting next combination based on previous data.
        - It is an algorithm that decides which hyperparameter should be evaluated next.
        - Optuna used Tree Structured parzen estimator (TPE) by default, but also support other sampling methods.


In [1]:
%pip install optuna

In [2]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# define objective function
def objective(trial):
    # suggest values for hyperparameters (search space)
    n_estimators = trial.suggest_int("n_estimators",50,200)
    max_depth = trial.suggest_int("max_depth",3,20)
    
    # create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model,X_train,y_train,cv=3,scoring="accuracy").mean()
    # return the accuracy score for optuna to maximize
    return score

In [6]:
# create a study object and optimize the objective function
study = optuna.create_study(study_name="My Study",direction="maximize",sampler=optuna.samplers.TPESampler()) 
study.optimize(objective,n_trials=50) 

[I 2026-05-30 14:30:00,330] A new study created in memory with name: My Study
[I 2026-05-30 14:30:00,705] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 95, 'max_depth': 9}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-05-30 14:30:00,967] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 68, 'max_depth': 8}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-05-30 14:30:01,570] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 153, 'max_depth': 19}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-05-30 14:30:01,927] Trial 3 finished with value: 0.7560521415270017 and parameters: {'n_estimators': 93, 'max_depth': 9}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-05-30 14:30:02,534] Trial 4 finished with value: 0.7579143389199254 and parameters: {'n_estimators': 157, 'max_depth': 9}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-05-30 14:30:02,774] T

In [7]:
print(f"Best Trail accuracy: {study.best_value}")
print(f"Best hyperparameters: {study.best_params}")

Best Trail accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 123, 'max_depth': 17}


In [8]:
from sklearn.metrics import accuracy_score

# train model using best hyperparameter from optuna
best_model = RandomForestClassifier(**study.best_params,random_state=42)

# fit model
best_model.fit(X_train,y_train)

# predictions
y_pred = best_model.predict(X_test)

# accuracy
acc = accuracy_score(y_test,y_pred)
print(f"test accuracy :{acc:.2f}")

test accuracy :0.74


Sampler in Optuna

In [9]:
# define objective function
def objective(trial):
    # suggest values for hyperparameters (search space)
    n_estimators = trial.suggest_int("n_estimators",50,200)
    max_depth = trial.suggest_int("max_depth",3,20)
    
    # create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model,X_train,y_train,cv=3,scoring="accuracy").mean()
    # return the accuracy score for optuna to maximize
    return score

In [10]:
# create a study object and optimize the objective function
study = optuna.create_study(study_name="My Study",direction="maximize",sampler=optuna.samplers.RandomSampler()) 
study.optimize(objective,n_trials=50) 

[I 2026-05-30 14:30:25,080] A new study created in memory with name: My Study
[I 2026-05-30 14:30:25,406] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 80, 'max_depth': 11}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-05-30 14:30:25,826] Trial 1 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 104, 'max_depth': 13}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-05-30 14:30:26,593] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 198, 'max_depth': 13}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-05-30 14:30:26,918] Trial 3 finished with value: 0.756052141527002 and parameters: {'n_estimators': 91, 'max_depth': 3}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-05-30 14:30:27,666] Trial 4 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 195, 'max_depth': 9}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-05-30 14:30:27,896]

In [11]:
print(f"Best Trail accuracy: {study.best_value}")
print(f"Best hyperparameters: {study.best_params}")

Best Trail accuracy: 0.7783985102420857
Best hyperparameters: {'n_estimators': 55, 'max_depth': 16}


In [12]:
# train model using best hyperparameter from optuna
best_model = RandomForestClassifier(**study.best_params,random_state=42)

# fit model
best_model.fit(X_train,y_train)

# predictions
y_pred = best_model.predict(X_test)

# accuracy
acc = accuracy_score(y_test,y_pred)
print(f"test accuracy :{acc:.2f}")

test accuracy :0.74


In [13]:
# define objective function
def objective(trial):
    # suggest values for hyperparameters (search space)
    n_estimators = trial.suggest_int("n_estimators",50,200)
    max_depth = trial.suggest_int("max_depth",3,20)
    
    # create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model,X_train,y_train,cv=3,scoring="accuracy").mean()
    # return the accuracy score for optuna to maximize
    return score

In [14]:
# create a study object and optimize the objective function
seach_space = {
    "n_estimators":[50,100,150,200],
    "max_depth":[5,10,15,20]
}
study = optuna.create_study(study_name="My Study",direction="maximize",sampler=optuna.samplers.GridSampler(search_space=seach_space)) 
study.optimize(objective,n_trials=50) 

[I 2026-05-30 14:30:51,216] A new study created in memory with name: My Study
[I 2026-05-30 14:30:51,595] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-05-30 14:30:52,170] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-05-30 14:30:52,378] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-05-30 14:30:52,795] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-05-30 14:30:53,318] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-05-30 14:30:53,5

In [15]:
print(f"Best Trail accuracy: {study.best_value}")
print(f"Best hyperparameters: {study.best_params}")

Best Trail accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [16]:
# train model using best hyperparameter from optuna
best_model = RandomForestClassifier(**study.best_params,random_state=42)

# fit model
best_model.fit(X_train,y_train)

# predictions
y_pred = best_model.predict(X_test)

# accuracy
acc = accuracy_score(y_test,y_pred)
print(f"test accuracy :{acc:.2f}")

test accuracy :0.74


Optuna Visualization

In [17]:
from optuna.visualization import (plot_optimization_history,
                                  plot_parallel_coordinate,
                                  plot_slice,
                                  plot_contour,
                                  plot_param_importances)

In [18]:
fig = plot_optimization_history(study)
fig.show()

In [19]:
print(len(study.trials))
print(study.best_params)

16
{'n_estimators': 50, 'max_depth': 5}


In [20]:
plot_parallel_coordinate(study).show()